# NAMRC N0 — Full-print energy audit (R8.4)

**Actual Rust thermal simulations**, not surrogate/analytical estimates. Runs full 100-layer B01 at common **dt=0.05 s** for A0, A2, A6, R2; audits heat input, energy storage, convection, radiation, heated-bed exchange, and residual. Computes clock-time diagnostics from the archived R8.4 source and activation file.

**Important:** Results only exist after the Rust simulations finish. No precomputed full-print N0 numbers are supplied. Full-resolution output for all 51,204 elements may be memory-prohibitive; start with `0.25 K` event-storage threshold and try `0.1 K` if possible. Earlier 4-layer, 0 K-threshold results cannot be compared directly.

Keep this notebook **next to** `n0_runner.py`, `n0_energy_audit.py`, `b01_solver.gcode`, and the `solver/` folder. Run in a local **Windows Jupyter** session.

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess
import pandas as pd

HERE = Path.cwd()
print('Working directory:', HERE)
for f in ['n0_runner.py','n0_energy_audit.py','b01_solver.gcode','solver/Cargo.toml']:
    assert (HERE/f).exists(), f'Missing {f}. Extract ZIP and open this notebook from its folder.'
print('Python:',sys.version.split()[0]);print('Cargo:',shutil.which('cargo'))
if shutil.which('cargo') is None:
    print('INSTALL RUST FIRST: https://rustup.rs/ (and MSVC Build Tools on Windows); restart Anaconda/Jupyter.')

## Step 1 — Verify files and build the original R8.4 Rust solver

The CARGO command installs **nothing** unless you already installed Rust. If you use a corporate Windows PC, confirm build tools are permitted. This is real solver compilation.

In [ ]:
import n0_runner as N0
N0.check()
N0.compile_solver()  # requires Rust/Cargo, creates solver/target/release/DES_thermal_simulation.exe
print('Built:',N0.binary())

## Step 2 — Four full-print ablation runs at **identical output threshold: 0.25 K**

A0 corrected; A2 only grouped D1–D3 legacy timing; A6 all original behaviors; R2 original except corrected timing. Keep the saved case directories—large raw histories are required for the independent audit. Runs and audits are restart-safe (successful cases and audits skipped). If a job crashes, inspect its `solver_stdout.log`.

In [ ]:
CASES = ['A0','A2','A6','R2']
MAIN_THRESHOLD = 0.25
for case in CASES:
    d=N0.prepare(case, MAIN_THRESHOLD)
    state=N0.run_case(d)
    print(case, 'simulation exit',state['returncode'], 'wall_s',state['wall_s'])
    record=N0.audit_case(d,case,MAIN_THRESHOLD)
    print(case,'energy residual (% of heat input):',round(100*record['audit']['rel_err_vs_heat_in'],4))
N0.summary()

## Step 3 — Review actual N0 results

In the summary, **net** bed heat transfer may be negative; `stored_change_since_deposition_J` typically negative. The signed residual is `stored_change_since_deposition + natural convection + radiation + bed loss`. *Do not call these FLIR-validation errors.*

In [ ]:
summary_path = HERE/'N0_results'/'N0_full_print_summary.csv'
assert summary_path.exists(), 'Run Step 2 first.'
df = pd.read_csv(summary_path)
cols=['case','storage_threshold_K','n_elements','physical_deposition_s',
      'integrated_deposition_s','deposition_clock_ratio',
      'deposited_thermal_excess_J','stored_change_since_deposition_J',
      'natural_convection_J','radiation_J','bed_net_J',
      'signed_energy_balance_residual_J','abs_residual_pct_heat_input']
display(df[cols].sort_values(['storage_threshold_K','case']))

## Step 4 — Optional finer output threshold: 0.1 K

Only run this if Step 2 is successful and your Windows machine has enough memory and disk space. Earlier full-print 0.1 K runs needed considerably more memory. **Do not attempt `0 K` full-resolution histories without a memory/disk plan.**

In [ ]:
RUN_FINER = False  # change to True after reviewing 0.25 K results
if RUN_FINER:
    for case in CASES:
        d=N0.prepare(case,0.1)
        state=N0.run_case(d)
        record=N0.audit_case(d,case,0.1)
        print(case,'0.1K residual (%)',round(100*record['audit']['rel_err_vs_heat_in'],4))
    N0.summary()
else:
    print('Skipped optional 0.1 K runs. Set RUN_FINER=True when resources permit.')

## Step 5 — Storage sensitivity and export

The threshold comparison quantifies uncertainty from the **event-driven output storage**, not physical observation uncertainty. A full-print energy percentage should be reported together with its threshold. Audit should show why the legacy timing produces disagreement under the true physical clock.

In [ ]:
N0.summary()
sens=HERE/'N0_results'/'N0_storage_sensitivity.csv'
if sens.exists():display(pd.read_csv(sens))
else:print('Storage comparison requires both 0.25 K and 0.1 K runs.')

# Create small portable results ZIP, excluding multi-GB solver histories.
from zipfile import ZipFile, ZIP_DEFLATED
out=HERE/'N0_results_for_review.zip'
with ZipFile(out,'w',ZIP_DEFLATED) as z:
    for p in (HERE/'N0_results').glob('*'):
        if p.suffix in ['.csv','.json']:z.write(p,p.relative_to(HERE))
    for p in (HERE/'N0_runs').glob('*/case_manifest.json'):
        z.write(p,p.relative_to(HERE))
    for p in (HERE/'N0_runs').glob('*/run_status.json'):
        z.write(p,p.relative_to(HERE))
print('Share with me:',out)
print('Raw histories remain in N0_runs/ (not included in review ZIP).')